# Load to Bronze Orders

In [0]:
bronze_orders_df = spark.read.table('retial_project.bronze.bronze_orders')
bronze_customers_df = spark.read.table('retial_project.bronze.bronze_customers')

# Data Cleaning and Data Transformation

##1. Fixed date format

In [0]:
from pyspark.sql.functions import try_to_date, col, coalesce

silver_df = bronze_orders_df.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "dd-MM-yyyy"),
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "dd/MM/yyyy")
    )
)
silver_df.display()

##2. Standardize Category Columns

In [0]:
silver_df.select('category').distinct().display()


In [0]:
from pyspark.sql.functions import upper, trim,when

silver_df = silver_df.withColumn(
    "category",
    upper(trim(col("category")))
)


silver_df = silver_df.withColumn(
        'category',when(col('category')=='ELECTRONIC','ELECTRONICS').
        when(col('category')=='HOME AND LIVING','HOME & LIVING').
        when(col('category')=='FASHON','FASHION').
        otherwise(col('category')))

##3. Remove Duplicate Orders


In [0]:
silver_df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()


In [0]:
from pyspark.sql.functions import count, countDistinct


silver_df.select(count('order_id'), countDistinct('order_id')).show()

In [0]:
silver_df = silver_df.dropDuplicates(['order_id'])

silver_df.select(count('order_id'), countDistinct('order_id')).show()

##5. Handle Null Values discount

### handle Null Values discount

In [0]:
from pyspark.sql.functions import *

silver_df = silver_df.withColumn('discount',
when(col('discount').isNull(), '0').when(col('discount') == 'ten', '01').otherwise(col('discount')))

### Convert Discount from 5% to Decimal

some rows contain

5%
we need 0.05

In [0]:

from pyspark.sql.functions import regexp_replace

silver_df = silver_df.withColumn(
    "discount",
    when(col("discount").like("5%"), 
           (regexp_replace("discount", "5%", "0.05"))
    ).when(col("discount").like("10%"), 
           (regexp_replace("discount", "10%", "0.1"))
    ).otherwise(col("discount"))
)

silver_df.display()
     

##6. Remove Negative quantity


In [0]:
print(type(silver_df))

In [0]:
silver_df = silver_df.filter(col('quantity') >= 0)
silver_df.display()

##7. Cast unit_price to Decimal


In [0]:
# silver_df.select('unit_price').display()
from pyspark.sql.types import *
silver_df = silver_df.withColumn('unit_price', 
                     col('unit_price').try_cast(DecimalType(10,2)))

##9. Recalculate Total Amount

In [0]:
silver_df = silver_df.withColumn('total_amount', 
                        col('quantity').cast('int') * col('unit_price') * (1 - col('discount').cast('double'))       
)

silver_df.display()
                     

In [0]:
silver_df.write.format('delta').mode('overwrite').saveAsTable('retial_project.silver.silver_orders')